In [1]:
# autoreload
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
import numpy as np
import pandas as pd
from tqdm import tqdm

In [3]:
mimic_iv_path = "/mnt/data/yihua/master/datasets/mimic-iv/2.2"
mm_dir = "/mnt/data/yihua/master/datasets/mimic-iv_processed"

output_dir = os.path.join(mm_dir, "preprocessing")
os.makedirs(output_dir, exist_ok=True)

In [5]:
# mm_dir = "/cis/home/charr165/Documents/multimodal"
# output_dir = os.path.join(mm_dir, "preprocessing")

# vitals_ts_df = pd.read_pickle(os.path.join(output_dir, "ts_vitals_icu.pkl"))
# procedureevents_ts_df = pd.read_pickle(os.path.join(output_dir, "ts_procedureevents_icu.pkl"))
# ts_labs_icu = pd.read_pickle(os.path.join(output_dir, "ts_labs_icu.pkl"))

# labs_vitals_ts_df = pd.read_pickle(os.path.join(output_dir, "ts_labs_vitals_icu.pkl"))
labs_vitals_ts_df = pd.read_pickle(os.path.join(output_dir, "ts_labs_vitals.pkl"))
labs_vitals_ts_df = labs_vitals_ts_df[labs_vitals_ts_df['icu_time_delta'] > 0]
labs_vitals_ts_df.drop(columns=['hosp_time_delta'], inplace=True)
labs_vitals_ts_df.rename(columns={'icu_time_delta': 'timedelta'}, inplace=True)
labs_vitals_ts_df.to_pickle(os.path.join(output_dir, "ts_labs_vitals_icu.pkl"))

In [13]:
labs_vitals_ts_df.dtypes

subject_id                 Int64
hadm_id                    Int64
stay_id                   object
timedelta                 object
Anion Gap                float64
Bicarbonate              float64
Calcium, Total           float64
Chloride                 float64
Creatinine               float64
Diastolic BP             float64
GCS - Eye Opening        float64
GCS - Motor Response     float64
GCS - Verbal Response    float64
Glucose                  float64
Heart Rate               float64
Hematocrit               float64
Hemoglobin               float64
MCH                      float64
MCHC                     float64
MCV                      float64
Magnesium                float64
Mean BP                  float64
Neutrophils              float64
O2 Saturation            float64
Phosphate                float64
Platelet Count           float64
RDW                      float64
Red Blood Cells          float64
Respiratory Rate         float64
Sodium                   float64
Systolic B

In [19]:
def impute_values_optimized(df):
    interval_length = 1
    global_means = df.mean()
    grouped = df.groupby('stay_id')
    all_stays_imputed = []

    for stay_id, group in tqdm(grouped):
        group = group.copy()
        
        group['time_interval'] = (group['timedelta'] / interval_length).astype(int)

        curr_stay_imputed = group.groupby('time_interval').last()

        curr_stay_imputed = curr_stay_imputed.reindex(range(curr_stay_imputed.index.max() + 1)).ffill()
        curr_stay_imputed['timedelta'] = curr_stay_imputed.index * interval_length
        curr_stay_imputed.iloc[:, 4:] = curr_stay_imputed.iloc[:, 4:].fillna(global_means, inplace=False)

        all_stays_imputed.append(curr_stay_imputed)

    imputed_df = pd.concat(all_stays_imputed, axis=0, ignore_index=True)

    return imputed_df

# imputed_vitals = impute_values_optimized(vitals_ts_df)
imputed_labs_vitals = impute_values_optimized(labs_vitals_ts_df)


100%|██████████| 73171/73171 [15:51<00:00, 76.90it/s] 


In [20]:
imputed_labs_vitals.to_pickle(os.path.join(output_dir, "imputed_ts_labs_vitals_icu.pkl"))
# imputed_labs_vitals.to_pickle(os.path.join(output_dir, "imputed_ts_labs_vitals.pkl"))
# imputed_vitals.to_pickle(os.path.join(output_dir, "imputed_ts_vitals_icu.pkl"))